# Modulo 4 · Analisis exploratorio de datos (EDA) y KPIs

Este es el corazon del BI: convertir datos crudos en **indicadores** (KPIs)
que respondan preguntas de negocio. Vamos a usar `groupby`, `pivot_table` y
agregaciones para construir los indicadores tipicos de un reporte de ventas.

**Contenidos:**
1. Estadistica descriptiva rapida
2. `groupby`: agregar datos por categoria
3. `pivot_table`: tablas cruzadas (como en Excel)
4. KPIs de negocio: ingreso total, ticket promedio, margen, top productos
5. Series de tiempo: tendencia mensual


In [1]:
import pandas as pd

df = pd.read_csv("../data/ventas_limpio.csv", parse_dates=["fecha"])
df.shape


(12030, 19)

## 1. Estadistica descriptiva rapida

In [2]:
df[["unidades", "precio_unitario", "ingreso", "costo", "utilidad", "margen_pct"]].describe().round(2)


,unidades,precio_unitario,ingreso,costo,utilidad,margen_pct
count,12030.0,12030.00,12030.00,12030.00,12030.00,12030.00
mean,4.0,34652.67,127579.38,82546.23,45033.15,0.35
std,2.0,36139.33,160976.62,103925.21,58589.69,0.06
min,1.0,1700.00,1456.00,1080.00,364.00,0.25
25%,2.0,9500.00,28191.00,18216.00,9492.00,0.29
50%,4.0,17840.00,62950.00,40761.00,21563.00,0.37
75%,6.0,46267.50,161617.25,103810.50,55875.00,0.40
max,7.0,172360.00,1131200.00,694260.00,452480.00,0.40


In [3]:
# Correlacion entre variables numericas
df[["unidades", "precio_unitario", "descuento_pct", "ingreso", "utilidad"]].corr().round(2)


,unidades,precio_unitario,descuento_pct,ingreso,utilidad
unidades,1.00,-0.02,-0.01,0.39,0.38
precio_unitario,-0.02,1.00,0.00,0.81,0.78
descuento_pct,-0.01,0.00,1.00,-0.06,-0.17
ingreso,0.39,0.81,-0.06,1.00,0.98
utilidad,0.38,0.78,-0.17,0.98,1.00


## 2. `groupby`: agregar datos por categoria

In [4]:
# Ingreso total por region
ingreso_por_region = df.groupby("region")["ingreso"].sum().sort_values(ascending=False)
ingreso_por_region


region
Sur       316719616.0
Centro    311224430.0
Este      311171395.0
Norte     309261637.0
Oeste     286402885.0
Name: ingreso, dtype: float64

In [5]:
# Multiples metricas a la vez con .agg()
resumen_region = df.groupby("region").agg(
    ingreso_total=("ingreso", "sum"),
    utilidad_total=("utilidad", "sum"),
    unidades_vendidas=("unidades", "sum"),
    ticket_promedio=("ingreso", "mean"),
    num_ventas=("id_venta", "count"),
).round(0).sort_values("ingreso_total", ascending=False)

resumen_region


,ingreso_total,utilidad_total,unidades_vendidas,ticket_promedio,num_ventas
region,,,,,
Sur,316719616.0,111388708.0,9811,129010.0,2455
Centro,311224430.0,109898468.0,9721,129623.0,2401
Este,311171395.0,109759435.0,9529,130580.0,2383
Norte,309261637.0,109337215.0,9819,127531.0,2425
Oeste,286402885.0,101365015.0,9273,121049.0,2366


In [6]:
# Agrupar por mas de una columna: categoria y region
resumen_categoria_region = df.groupby(["categoria", "region"])["ingreso"].sum().unstack()
resumen_categoria_region.round(0)


region,Centro,Este,Norte,Oeste,Sur
categoria,,,,,
Deportes,87336866.0,88232704.0,84121492.0,71356516.0,88379538.0
Electronica,25682026.0,25394526.0,26448755.0,22976483.0,25058034.0
Hogar,61816063.0,68506992.0,63697879.0,57712357.0,67708944.0
Oficina,100118394.0,92506151.0,93523171.0,95135039.0,93388972.0
Ropa,36271081.0,36531022.0,41470340.0,39222490.0,42184128.0


## 3. `pivot_table`: tablas cruzadas

In [7]:
# pivot_table es equivalente a groupby + unstack, pero mas directo
tabla_pivote = pd.pivot_table(
    df,
    values="ingreso",
    index="categoria",
    columns="region",
    aggfunc="sum",
    fill_value=0,
    margins=True,        # agrega totales por fila y columna
    margins_name="Total",
)
tabla_pivote.round(0)


region,Centro,Este,Norte,Oeste,Sur,Total
categoria,,,,,,
Deportes,87336866.0,88232704.0,84121492.0,71356516.0,88379538.0,4.194271e+08
Electronica,25682026.0,25394526.0,26448755.0,22976483.0,25058034.0,1.255598e+08
Hogar,61816063.0,68506992.0,63697879.0,57712357.0,67708944.0,3.194422e+08
Oficina,100118394.0,92506151.0,93523171.0,95135039.0,93388972.0,4.746717e+08
Ropa,36271081.0,36531022.0,41470340.0,39222490.0,42184128.0,1.956791e+08
Total,311224430.0,311171395.0,309261637.0,286402885.0,316719616.0,1.534780e+09


## 4. KPIs de negocio

In [8]:
ingreso_total = df["ingreso"].sum()
utilidad_total = df["utilidad"].sum()
margen_promedio = df["utilidad"].sum() / df["ingreso"].sum()
ticket_promedio = df["ingreso"].mean()
num_ventas = len(df)
unidades_totales = df["unidades"].sum()

print(f"Ingreso total:      ${ingreso_total:,.0f}")
print(f"Utilidad total:     ${utilidad_total:,.0f}")
print(f"Margen promedio:    {margen_promedio:.1%}")
print(f"Ticket promedio:    ${ticket_promedio:,.0f}")
print(f"Numero de ventas:   {num_ventas:,}")
print(f"Unidades vendidas:  {unidades_totales:,}")


Ingreso total:      $1,534,779,963
Utilidad total:     $541,748,841
Margen promedio:    35.3%
Ticket promedio:    $127,579
Numero de ventas:   12,030
Unidades vendidas:  48,153


In [9]:
# Top 5 productos por ingreso
top_productos = (
    df.groupby("producto")["ingreso"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
top_productos


producto
Bicicleta           246276178.0
Escritorio          178287150.0
Silla Ergonomica    152000289.0
Aspiradora          139826843.0
Impresora           133656506.0
Name: ingreso, dtype: float64

In [10]:
# Top 5 vendedores por utilidad generada
top_vendedores = (
    df.groupby("vendedor")["utilidad"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
top_vendedores


vendedor
Gabriela Nunez    57178810.0
Javier Castro     57108755.0
Carla Diaz        56258116.0
Felipe Soto       55669809.0
Diego Rojas       53489838.0
Name: utilidad, dtype: float64

In [11]:
# Participacion de mercado (% del ingreso) por categoria
participacion = (df.groupby("categoria")["ingreso"].sum() / ingreso_total * 100).round(1)
participacion.sort_values(ascending=False)


categoria
Oficina        30.9
Deportes       27.3
Hogar          20.8
Ropa           12.7
Electronica     8.2
Name: ingreso, dtype: float64

## 5. Series de tiempo: tendencia mensual

In [12]:
# Ingreso mensual: usamos dt.to_period para agrupar por mes-anio
df["periodo"] = df["fecha"].dt.to_period("M")
ingreso_mensual = df.groupby("periodo")["ingreso"].sum()
ingreso_mensual.head(12)


periodo
2023-01    63190998.0
2023-02    55313968.0
2023-03    48515763.0
2023-04    44634813.0
2023-05    37823008.0
2023-06    27128794.0
2023-07    21248137.0
2023-08    23190468.0
2023-09    27587761.0
2023-10    42815900.0
2023-11    55213905.0
2023-12    56737642.0
Freq: M, Name: ingreso, dtype: float64

In [13]:
# Variacion porcentual mes a mes
variacion_mensual = ingreso_mensual.pct_change().round(4) * 100
variacion_mensual.tail(12)


periodo
2025-01     8.01
2025-02   -11.82
2025-03    -5.57
2025-04   -21.61
2025-05   -15.86
2025-06   -25.05
2025-07   -33.83
2025-08    43.83
2025-09     2.82
2025-10    36.20
2025-11    43.81
2025-12    21.00
Freq: M, Name: ingreso, dtype: float64

## 🧠 Retos del modulo 4

1. Calcula el ingreso total, la utilidad total y el margen promedio **por categoria**.
2. ¿Cual es el `segmento_cliente` que genera mas utilidad total?
3. Construye una `pivot_table` que muestre el ticket promedio (`ingreso` promedio)
   por `producto` (filas) y `segmento_cliente` (columnas).
4. ¿En que mes del dataset se registro el mayor ingreso? ¿Y el menor?
5. Calcula, para cada vendedor, cuantas ventas hizo y cual fue su margen promedio.
   Ordena de mayor a menor margen promedio.

Compara tus resultados con `solutions/04_analisis_exploratorio_solucion.py`.


In [14]:
# Escribe aqui tu solucion


